## Transform Ciruits Data
- Read bronze circuits table
- Keep only the columns required for analytics(Drop url column)
- Standardise column name using snakecase(circuitsId->circuits_id)
- Rename columns to make them more powerful(lat->latitiude)
- Filter out rows where circuitsId is null (business key validation)
- Remove duplicate records.
- Transform values of column circuit_name and locaity to Title Case
- Write the transformed data to silver circuits table

Below changes are required to implement incremental processing
- 1.Accept batch_id as parameter to notebook
- 2.Process data for only batch_id being passed in(filter reading bronze using batch id)
- 3.Add created_timestamp,updated_timestamp,batch_id to silver table
- 4.Merge processed data to silver table
- created_timestamp sould only be populated at the time of inserting/creating the record .It should not be updated     during merge update
- Ensure that we are not overwriting data in silver table by older bronze data (re run scenrio)

In [0]:
dbutils.widgets.text("p_batch_id","")
v_batch_id=dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/03.silver-help

In [0]:
bronze_table=f"{catalog_name}.{bronze_schema}.circuits"
silver_table=f"{catalog_name}.{silver_schema}.circuits"

### Step 1 Read bronze circuits table

In [0]:
import pyspark.sql.functions as F

In [0]:
circuits_df=(
    spark.table(bronze_table).filter(F.col("batch_id") == v_batch_id)
)


### Srep 2 -Keep only the columns required for analytics(Drop url column)

In [0]:
circuits_df.select("circuitId",
                   "circuitName",
                   "lat",
                   "long",
                   "locality",
                   "country",
                   "ingestion_timestamp",
                   "source_file")

DataFrame[circuitId: string, circuitName: string, lat: double, long: double, locality: string, country: string, ingestion_timestamp: timestamp, source_file: string]

In [0]:
#flexible   `

circuitsselected_df=circuits_df.select(F.col("circuitId"),
                   F.col("circuitName"),
                   F.col("lat"),
                   F.col("long"),
                   F.col("locality"),
                   F.col("country"),
                   F.col("ingestion_timestamp"),
                   F.col("source_file"),
                   F.col("batch_id"))
               

### Step 3 & 4 - Stamdardise column name
- Standardise column name using snakecase(circuitsId->circuits_id)
- Rename columns to make them more powerful(lat->latitiude)

In [0]:
circuits_renamed_df=(
    circuitsselected_df.withColumnRenamed("circuitId","circuit_id")
                       .withColumnRenamed("circuitName","circuit_name")
                       .withColumnRenamed("lat","latitude")
                       .withColumnRenamed("long","longitude")
                       
)


In [0]:
#different method multiple columns #not supported here
#circuits_renamed_df=(
   # circuitsselected_df.withcolumnsRenamed({"circuitId":"circuit_id",
                      # "circuitName":"circuit_name",
                       #"lat":"latitude",
                     #  "long":"longitude"})
#)

### Step 5 Filter out rows where circuitsId is null (business key validation)

In [0]:
circuits_valid_df=circuits_renamed_df.filter("circuit_id IS NOT NULL")
display(circuits_valid_df)

circuit_id,circuit_name,latitude,longitude,locality,country,ingestion_timestamp,source_file,batch_id
adelaide,adelaide street circuit,-34.9272,138.617,adelaide,Australia,2026-10-20T05:48:50.737Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/circuits.csv,2025-01
ain-diab,ain diab,33.5786,-7.6875,casablanca,Morocco,2026-10-20T05:48:50.737Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/circuits.csv,2025-01
aintree,aintree,53.4769,-2.94056,liverpool,UK,2026-10-20T05:48:50.737Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/circuits.csv,2025-01
albert_park,albert park grand prix circuit,-37.8497,144.968,melbourne,Australia,2026-10-20T05:48:50.737Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/circuits.csv,2025-01
americas,circuit of the americas,30.1328,-97.6411,austin,USA,2026-10-20T05:48:50.737Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/circuits.csv,2025-01
anderstorp,scandinavian raceway,57.2653,13.6042,anderstorp,Sweden,2026-10-20T05:48:50.737Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/circuits.csv,2025-01
avus,avus,52.4806,13.2514,berlin,Germany,2026-10-20T05:48:50.737Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/circuits.csv,2025-01
bahrain,bahrain international circuit,26.0325,50.5106,sakhir,Bahrain,2026-10-20T05:48:50.737Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/circuits.csv,2025-01
baku,baku city circuit,40.3725,49.8533,baku,Azerbaijan,2026-10-20T05:48:50.737Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/circuits.csv,2025-01
boavista,circuito da boavista,41.1705,-8.67325,oporto,Portugal,2026-10-20T05:48:50.737Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/circuits.csv,2025-01


In [0]:
#using methods
circuits_valid_df=circuits_renamed_df.filter(F.col("circuit_id").isNotNull())
display(circuits_valid_df)


circuit_id,circuit_name,latitude,longitude,locality,country,ingestion_timestamp,source_file,batch_id
adelaide,adelaide street circuit,-34.9272,138.617,adelaide,Australia,2026-10-20T05:48:50.737Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/circuits.csv,2025-01
ain-diab,ain diab,33.5786,-7.6875,casablanca,Morocco,2026-10-20T05:48:50.737Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/circuits.csv,2025-01
aintree,aintree,53.4769,-2.94056,liverpool,UK,2026-10-20T05:48:50.737Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/circuits.csv,2025-01
albert_park,albert park grand prix circuit,-37.8497,144.968,melbourne,Australia,2026-10-20T05:48:50.737Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/circuits.csv,2025-01
americas,circuit of the americas,30.1328,-97.6411,austin,USA,2026-10-20T05:48:50.737Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/circuits.csv,2025-01
anderstorp,scandinavian raceway,57.2653,13.6042,anderstorp,Sweden,2026-10-20T05:48:50.737Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/circuits.csv,2025-01
avus,avus,52.4806,13.2514,berlin,Germany,2026-10-20T05:48:50.737Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/circuits.csv,2025-01
bahrain,bahrain international circuit,26.0325,50.5106,sakhir,Bahrain,2026-10-20T05:48:50.737Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/circuits.csv,2025-01
baku,baku city circuit,40.3725,49.8533,baku,Azerbaijan,2026-10-20T05:48:50.737Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/circuits.csv,2025-01
boavista,circuito da boavista,41.1705,-8.67325,oporto,Portugal,2026-10-20T05:48:50.737Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/circuits.csv,2025-01


### Step -6 Remove duplicates

In [0]:
#circuits_distinct_df=circuits_valid_df.distinct()
#display(circuits_distinct_df)

In [0]:
#dropduplicates - you can give column name and distinct will be applied on that particular column only
circuits_distinct_df=circuits_valid_df.dropDuplicates(["Circuit_id"])
display(circuits_distinct_df)

circuit_id,circuit_name,latitude,longitude,locality,country,ingestion_timestamp,source_file,batch_id
adelaide,adelaide street circuit,-34.9272,138.617,adelaide,Australia,2026-10-20T05:48:50.737Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/circuits.csv,2025-01
ain-diab,ain diab,33.5786,-7.6875,casablanca,Morocco,2026-10-20T05:48:50.737Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/circuits.csv,2025-01
aintree,aintree,53.4769,-2.94056,liverpool,UK,2026-10-20T05:48:50.737Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/circuits.csv,2025-01
albert_park,albert park grand prix circuit,-37.8497,144.968,melbourne,Australia,2026-10-20T05:48:50.737Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/circuits.csv,2025-01
americas,circuit of the americas,30.1328,-97.6411,austin,USA,2026-10-20T05:48:50.737Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/circuits.csv,2025-01
anderstorp,scandinavian raceway,57.2653,13.6042,anderstorp,Sweden,2026-10-20T05:48:50.737Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/circuits.csv,2025-01
avus,avus,52.4806,13.2514,berlin,Germany,2026-10-20T05:48:50.737Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/circuits.csv,2025-01
bahrain,bahrain international circuit,26.0325,50.5106,sakhir,Bahrain,2026-10-20T05:48:50.737Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/circuits.csv,2025-01
baku,baku city circuit,40.3725,49.8533,baku,Azerbaijan,2026-10-20T05:48:50.737Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/circuits.csv,2025-01
boavista,circuito da boavista,41.1705,-8.67325,oporto,Portugal,2026-10-20T05:48:50.737Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/circuits.csv,2025-01


### Step -7 Transform values of column circuit_name and locality to Title Case

In [0]:
circuits_final_df=(
    circuits_distinct_df
    .withColumn("circuit_name",F.initcap(F.col("circuit_name")))
    .withColumn("locality",F.initcap(F.col("locality")))
)
display(circuits_final_df)

circuit_id,circuit_name,latitude,longitude,locality,country,ingestion_timestamp,source_file,batch_id
adelaide,Adelaide Street Circuit,-34.9272,138.617,Adelaide,Australia,2026-10-20T05:48:50.737Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/circuits.csv,2025-01
ain-diab,Ain Diab,33.5786,-7.6875,Casablanca,Morocco,2026-10-20T05:48:50.737Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/circuits.csv,2025-01
aintree,Aintree,53.4769,-2.94056,Liverpool,UK,2026-10-20T05:48:50.737Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/circuits.csv,2025-01
albert_park,Albert Park Grand Prix Circuit,-37.8497,144.968,Melbourne,Australia,2026-10-20T05:48:50.737Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/circuits.csv,2025-01
americas,Circuit Of The Americas,30.1328,-97.6411,Austin,USA,2026-10-20T05:48:50.737Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/circuits.csv,2025-01
anderstorp,Scandinavian Raceway,57.2653,13.6042,Anderstorp,Sweden,2026-10-20T05:48:50.737Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/circuits.csv,2025-01
avus,Avus,52.4806,13.2514,Berlin,Germany,2026-10-20T05:48:50.737Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/circuits.csv,2025-01
bahrain,Bahrain International Circuit,26.0325,50.5106,Sakhir,Bahrain,2026-10-20T05:48:50.737Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/circuits.csv,2025-01
baku,Baku City Circuit,40.3725,49.8533,Baku,Azerbaijan,2026-10-20T05:48:50.737Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/circuits.csv,2025-01
boavista,Circuito Da Boavista,41.1705,-8.67325,Oporto,Portugal,2026-10-20T05:48:50.737Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/circuits.csv,2025-01


### Step-8 Write the transformed data to silver circuits table

In [0]:
# circuits_final_df=(
#     circuits_final_df
#        .withColumn("created_timestamp",F.current_timestamp())
#        .withColumn("updated_timestamp",F.current_timestamp())
# )

In [0]:
# from delta.tables import DeltaTable

# if not spark.catalog.tableExists(silver_table):
#     (
#         circuits_final_df
#         .write
#         .mode("overwrite")
#         .format("delta")
#         .saveAsTable(silver_table)
#     )
# else:
#     delta_table = DeltaTable.forName(spark, silver_table)
#     (
#         delta_table.alias("t")
#         .merge(
#             circuits_final_df.alias("s"),
#             "t.circuit_id = s.circuit_id"
#         )
#         .whenMatchedUpdate(
#             condition="s.batch_id >= t.batch_id",
#             set={
#                 "circuit_name": "s.circuit_name",
#                 "latitude": "s.latitude",
#                 "longitude": "s.longitude",
#                 "locality": "s.locality",
#                 "country": "s.country",
#                 "ingestion_timestamp": "s.ingestion_timestamp",
#                 "source_file": "s.source_file",
#                 "batch_id": "s.batch_id",
#                 "updated_timestamp": "s.updated_timestamp"
#             }
#         )
#         .whenNotMatchedInsertAll()
#         .execute()
#     )

In [0]:
write_to_silver(
    circuits_final_df,
    silver_table,
    merge_condition="t.circuit_id=s.circuit_id",
    columns_to_update=[
        "circuit_name",
        "latitude",
        "longitude",
        "locality",
        "country",
        "ingestion_timestamp",
        "source_file",
        "batch_id"

    ]
)

In [0]:
display(spark.read.table(silver_table))

circuit_id,circuit_name,latitude,longitude,locality,country,ingestion_timestamp,source_file,batch_id,created_timestamp,updated_timestamp
bahrain,Bahrain International Circuit,26.0325,50.5106,Sakhir,Bahrain,2026-10-20T05:48:50.737Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/circuits.csv,2025-01,2026-10-20T08:33:55.853Z,2026-10-20T11:10:44.122Z
detroit,Detroit Street Circuit,42.3298,-83.0401,Detroit,USA,2026-10-20T05:48:50.737Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/circuits.csv,2025-01,2026-10-20T08:33:55.853Z,2026-10-20T11:10:44.122Z
mosport,Mosport International Raceway,44.0481,-78.6756,Ontario,Canada,2026-10-20T05:48:50.737Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/circuits.csv,2025-01,2026-10-20T08:33:55.853Z,2026-10-20T11:10:44.122Z
red_bull_ring,Red Bull Ring,47.2197,14.7647,Spielberg,Austria,2026-10-20T05:48:50.737Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/circuits.csv,2025-01,2026-10-20T08:33:55.853Z,2026-10-20T11:10:44.122Z
sebring,Sebring International Raceway,27.4547,-81.3483,Florida,USA,2026-10-20T05:48:50.737Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/circuits.csv,2025-01,2026-10-20T08:33:55.853Z,2026-10-20T11:10:44.122Z
adelaide,Adelaide Street Circuit,-34.9272,138.617,Adelaide,Australia,2026-10-20T05:48:50.737Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/circuits.csv,2025-01,2026-10-20T08:33:55.853Z,2026-10-20T11:10:44.122Z
buddh,Buddh International Circuit,28.3487,77.5331,Uttar Pradesh,India,2026-10-20T05:48:50.737Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/circuits.csv,2025-01,2026-10-20T08:33:55.853Z,2026-10-20T11:10:44.122Z
monaco,Circuit De Monaco,43.7347,7.42056,Monte Carlo,Monaco,2026-10-20T05:48:50.737Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/circuits.csv,2025-01,2026-10-20T08:33:55.853Z,2026-10-20T11:10:44.122Z
pedralbes,Circuit De Pedralbes,41.3903,2.11667,Barcelona,Spain,2026-10-20T05:48:50.737Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/circuits.csv,2025-01,2026-10-20T08:33:55.853Z,2026-10-20T11:10:44.122Z
tremblant,Circuit Mont-tremblant,46.1877,-74.6099,Quebec,Canada,2026-10-20T05:48:50.737Z,dbfs:/Volumes/formula1_incr/landing/files/2025-01/circuits.csv,2025-01,2026-10-20T08:33:55.853Z,2026-10-20T11:10:44.122Z
